In [50]:
import sys
from pathlib import Path

# I had problems with relative imports in Jupyter notebooks, so I added the project root to sys.path
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Added to sys.path:", PROJECT_ROOT)

Added to sys.path: C:\Users\raula\Desktop\renewable\RES---Assignment-1


In [51]:
from pathlib import Path
import pandas as pd

from src.data_loader import load_rts24_from_csv
from src.models import Step1InputData, Step1MarketClearing, Step2InputData, Step2MarketClearing
from src.postprocess import step1_build_summary_tables, step2_build_summary_tables

Define data directory

In [52]:
DATA_DIR = Path("..") / "data"

data = load_rts24_from_csv(DATA_DIR)

print(f"Slack bus: {data.slack_bus}")
print(f"Number of generators: {len(data.generators)}")
print(f"Number of lines: {len(data.lines)}")

Slack bus: 13
Number of generators: 12
Number of lines: 34


# STEP 1

Choose hour for copper-plate market clearing

In [53]:
H = 19  # peak hour example

Extract hour-specific data

In [54]:
# Generators
gen_df = data.generators.copy()

GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))

# Wind availability
wind_farms_df = data.wind_farms.copy()
wind_av_h = data.wind_availability.query("hour == @H")

WINDS = wind_farms_df["wind_id"].tolist()
wind_avail = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))


# Nodal demand
nodal_load_h = data.nodal_load.query("hour == @H")

LOAD_BUSES = nodal_load_h.query("demand_MW > 0")["bus"].tolist()
demand_pmax = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))


# Demand bids
bids_h = data.demand_bids.query("hour == @H")
demand_bid = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

Build input object

In [55]:
inp = Step1InputData(
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    wind_avail=wind_avail,
    demand_pmax=demand_pmax,
    demand_bid=demand_bid,
)

Solve Step 1 (Copper Plate Market Clearing)

In [56]:
model = Step1MarketClearing(inp, output_flag=0)
model.run()

**Results**

In [57]:
df_gen, df_wind, df_dem, totals = step1_build_summary_tables(model.results, inp)

print("Market price:", round(totals["market_price"], 2), "USD/MWh")
print("Objective (Social Welfare):", round(totals["objective_welfare"], 2))
print("Total operating cost:", round(totals["total_operating_cost"], 2))

print("\nTotal generation:", round(totals["total_gen_MW"], 2), "MW")
print("Total wind:", round(totals["total_wind_MW"], 2), "MW")
print("Total demand served:", round(totals["total_demand_served_MW"], 2), "MW")

Market price: 13.32 USD/MWh
Objective (Social Welfare): 619868.95
Total operating cost: 16251.05

Total generation: 2169.19 MW
Total wind: 481.31 MW
Total demand served: 2650.5 MW


Generator results

In [58]:
df_gen

,generator,p_MW,marginal_cost,profit
0,1,99.185514,13.32,0.0
1,2,0.000000,13.32,0.0
2,3,0.000000,20.70,-0.0
3,4,0.000000,20.93,-0.0
4,5,0.000000,26.11,-0.0
5,6,155.000000,10.52,434.0
6,7,155.000000,10.52,434.0
7,8,400.000000,6.02,2920.0
8,9,400.000000,5.47,3140.0
9,10,300.000000,0.00,3996.0


Wind results

In [59]:
df_wind

,wind,p_MW,profit
0,1,73.622733,980.654803
1,2,83.964245,1118.403746
2,3,80.763464,1075.769341
3,4,82.379405,1097.293675
4,5,73.687627,981.519194
5,6,86.897011,1157.468190


Demand results

In [60]:
df_dem

,bus,d_MW,bid_price,utility
0,1,100.7190,240.0,22830.98292
1,2,90.1170,240.0,20427.72156
2,3,166.9815,240.0,37851.36642
3,4,68.9130,240.0,15621.19884
4,5,66.2625,240.0,15020.38350
5,6,127.2240,240.0,28839.13632
6,7,116.6220,240.0,26435.87496
7,8,159.0300,240.0,36048.92040
8,9,161.6805,240.0,36649.73574
9,10,180.2340,240.0,40855.44312


Welfare breakdown check

In [61]:
print("Total generator profit:", round(totals["total_gen_profit"], 2))
print("Total wind profit:", round(totals["total_wind_profit"], 2))
print("Total demand utility:", round(totals["total_demand_utility"], 2))

print("\nCheck: Utility - Cost = Welfare")
print(round(
    totals["total_demand_utility"]
    + totals["total_gen_profit"]
    + totals["total_wind_profit"],
    6
))

Total generator profit: 12642.5
Total wind profit: 6411.11
Total demand utility: 600815.34

Check: Utility - Cost = Welfare
619868.948948


# STEP 2

In [62]:
# Generators
gen_df = data.generators.copy()

GENERATORS = gen_df["unit"].tolist()
generator_cost = dict(zip(gen_df["unit"], gen_df["Ci_USD_per_MWh"]))
generator_pmax = dict(zip(gen_df["unit"], gen_df["Pmax_MW"]))

# Hours (all 24)
hours = [int(h) for h in sorted(data.wind_availability["hour"].unique())]

# Wind availability
wind_farms_df = data.wind_farms.copy()
WINDS = wind_farms_df["wind_id"].tolist()

wind_avail_t = {}

for h in hours:
    wind_av_h = data.wind_availability.query("hour == @h")
    wind_avail_t[h] = dict(zip(wind_av_h["wind_id"], wind_av_h["available_MW"]))


# Nodal demand
LOAD_BUSES = data.nodal_load.query("demand_MW > 0")["bus"].unique().tolist()

demand_pmax_t = {}
for h in hours:
    nodal_load_h = data.nodal_load.query("hour == @h")
    demand_pmax_t[h] = dict(zip(nodal_load_h["bus"], nodal_load_h["demand_MW"]))


# Demand bids
demand_bid_t = {}
for h in hours:
    bids_h = data.demand_bids.query("hour == @h")
    demand_bid_t[h] = dict(zip(bids_h["bus"], bids_h["bid_price_USD_per_MWh"]))

In [63]:
inpu = Step2InputData(
    GENERATORS=GENERATORS,
    WINDS=WINDS,
    LOAD_BUSES=LOAD_BUSES,
    generator_cost=generator_cost,
    generator_pmax=generator_pmax,
    wind_avail_t=wind_avail_t,
    demand_pmax_t=demand_pmax_t,
    demand_bid_t=demand_bid_t,
    hours=hours                   
)
print("hours:", hours, type(hours))


hours: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24] <class 'list'>


In [64]:
model2 = Step2MarketClearing(inpu, output_flag=0)
model2.run()

In [ ]:
df_gen, df_wind, df_dem, df_storage, df_prices, totals = step2_build_summary_tables(model2.results, inpu)


Avg market price: 10.42 USD/MWh
Objective (Social Welfare 24h): 11238148.2
Total operating cost 24h: 231095.4

Total generation 24h: 37703.4 MWh
Total wind 24h: 12506.05 MWh
Total demand served 24h: 52771.46 MWh
Total storage profit 24h: -25340.24

=== Generator Summary 24h ===
    generator  total_p_MWh  marginal_cost  total_profit_24h
0           1         5.19          13.32               0.0
1           2         0.00          13.32               0.0
2           3         0.00          20.70               0.0
3           4         0.00          20.93               0.0
4           5         0.00          26.11               0.0
5           6      2218.90          10.52            1007.5
6           7      2295.76          10.52            1007.5
7           8      9321.52           6.02           42200.0
8           9      9600.00           5.47           47480.0
9          10      7200.00           0.00           74994.0
10         11      5408.30          10.52            2015.0
1

In [66]:
print("\n" + "="*60)
print("STEP 2 - COPPER PLATE 24H WITH STORAGE")
print("="*60)

print("\n1. HOURLY PRICES")
print(df_prices.round(2))

print("\n2. TOTALS 24 HOURS")
print(pd.Series(totals).round(2))

print("\n3. GENERATOR TOTALS & PROFITS")
print(df_gen.round(2))

print("\n4. WIND TOTALS & PROFITS") 
print(df_wind.round(2))

print("\n5. STORAGE OPERATION (Hourly)")
print(df_storage.round(2))

print("\n6. DEMAND TOTALS & UTILITY")
print(df_dem.round(2))


STEP 2 - COPPER PLATE 24H WITH STORAGE

1. HOURLY PRICES
    hour  price
0      1   6.02
1      2  10.52
2      3  10.52
3      4  10.52
4      5  10.52
5      6  10.52
6      7  10.52
7      8  10.89
8      9  10.89
9     10  10.89
10    11  10.89
11    12  10.52
12    13  10.52
13    14  10.89
14    15  10.52
15    16  10.89
16    17  10.89
17    18  10.89
18    19  13.32
19    20  10.89
20    21  10.89
21    22  10.52
22    23  10.52
23    24   6.02

2. TOTALS 24 HOURS
objective_welfare_24h       11238148.20
total_operating_cost_24h      231095.40
total_gen_MWh                  37703.40
total_wind_MWh                 12506.05
total_demand_served_MWh        52771.46
total_gen_profit_24h          169554.50
total_wind_profit_24h         130368.63
total_demand_utility_24h    10912884.83
total_storage_profit_24h      -25340.24
avg_price                         10.42
max_price                         13.32
min_price                          6.02
dtype: float64

3. GENERATOR TOTALS & PROF